### STEP 0: Loading Data

In [25]:
df = pd.read_csv("Dataset.csv")

features = ["Runs", "Batting_Avg", "Strike_Rate", "Wickets", "Economy"]
A = np.array(df[features].values, dtype=float)  # Convert selected features into a NumPy matrix for linear algebra operations

print("Matrix Shape:", A.shape)
df.head()

Matrix Shape: (100, 5)


,Player_ID,Name,Role,Batting_Style,Bowling_Style,Matches,Runs,Batting_Avg,Strike_Rate,Wickets,Bowling_Avg,Economy,Fielding_Rating
0,1,Player_1,Batsman,Right-hand,NaN,66,1939,27.7,126.7,0,0.0,0.0,7
1,2,Player_2,Bowler,Right-hand,Spin,83,2100,16.5,105.7,111,32.1,6.4,8
2,3,Player_3,Wicketkeeper,Right-hand,NaN,120,3827,42.9,149.0,0,0.0,0.0,6
3,4,Player_4,Bowler,Left-hand,Medium,144,2615,23.6,107.3,129,31.2,7.3,10
4,5,Player_5,Wicketkeeper,Right-hand,NaN,135,2429,26.1,131.4,0,0.0,0.0,8


### STEP 1: Matrix Representation  
Converting real-world player data into a numerical matrix form.  
We extract selected features and store them as matrix A for linear algebra operations.

In [15]:
print("First 5 rows of Matrix A:")
pd.DataFrame(A[:5], columns=features)

First 5 rows of Matrix A:


,Runs,Batting_Avg,Strike_Rate,Wickets,Economy
0,1939.0,27.7,126.7,0.0,0.0
1,2100.0,16.5,105.7,111.0,6.4
2,3827.0,42.9,149.0,0.0,0.0
3,2615.0,23.6,107.3,129.0,7.3
4,2429.0,26.1,131.4,0.0,0.0


### STEP 2: RREF (Gaussian Elimination)  
RREF simplifies a matrix to identify independent columns.  
We compute RREF to find pivot columns and determine the rank of the matrix.

In [26]:
A_sym = Matrix(A[:10].tolist())  # Convert the first 10 rows to a SymPy Matrix for symbolic computation
rref_matrix, pivot_cols = A_sym.rref()  # Compute the Reduced Row Echelon Form and identify pivot column indices
rank = len(pivot_cols)  # The number of pivots determines the rank of the matrix

print("RREF:")
display(rref_matrix)

print("Pivot Columns:", pivot_cols)
print("Rank:", rank)

RREF:


Matrix([
[1, 0, 0, 0, 0],
[0, 1, 0, 0, 0],
[0, 0, 1, 0, 0],
[0, 0, 0, 1, 0],
[0, 0, 0, 0, 1],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]])

Pivot Columns: (0, 1, 2, 3, 4)
Rank: 5


### STEP 3: Vector Space Structure  
Vector spaces describe all possible linear combinations of data.  
We compute rank, nullity, and null space to understand feature dependencies.

In [27]:
nullity = A.shape[1] - rank  # Calculate dimension of null space using Rank-Nullity Theorem
null_basis = la.null_space(A)  # Compute the orthonormal basis for the null space of A

print("Rank:", rank)
print("Nullity:", nullity)
print("Null Space Basis:\n", null_basis)

Rank: 5
Nullity: 0
Null Space Basis:
 []


### STEP 4: Basis Selection  
A basis is a minimal set of independent vectors spanning a space.  
We select pivot columns to form the basis and remove redundant features.

In [28]:
basis_cols = list(pivot_cols)
basis = A[:, basis_cols]  # Extract columns corresponding to pivots to form a basis for the column space
independent_features = [features[i] for i in basis_cols]

print("Independent Features:", independent_features)
pd.DataFrame(basis[:5], columns=independent_features)

Independent Features: ['Runs', 'Batting_Avg', 'Strike_Rate', 'Wickets', 'Economy']


,Runs,Batting_Avg,Strike_Rate,Wickets,Economy
0,1939.0,27.7,126.7,0.0,0.0
1,2100.0,16.5,105.7,111.0,6.4
2,3827.0,42.9,149.0,0.0,0.0
3,2615.0,23.6,107.3,129.0,7.3
4,2429.0,26.1,131.4,0.0,0.0


### STEP 5: Gram-Schmidt Orthogonalization  
Orthogonalization makes vectors perpendicular (independent directions).  
We apply Gram-Schmidt to convert basis vectors into orthogonal vectors.

In [29]:
def gram_schmidt(vectors):
    orthogonal = []
    for v in vectors:
        w = v.copy().astype(float)
        for u in orthogonal:
            w -= (np.dot(v, u) / np.dot(u, u)) * u  # Subtract the projection of v onto previously found orthogonal vectors
        if np.linalg.norm(w) > 1e-10:  # Only add vectors that are not effectively zero
            orthogonal.append(w)
    return np.array(orthogonal)

ortho_basis = gram_schmidt(A[:5])

print("Orthogonal Basis:")
ortho_basis

Orthogonal Basis:


array([[ 1.93900000e+03,  2.77000000e+01,  1.26700000e+02,
         0.00000000e+00,  0.00000000e+00],
       [ 2.24244917e+00, -1.34679650e+01, -3.13736883e+01,
         1.11000000e+02,  6.40000000e+00],
       [ 6.19034924e+00, -8.36163817e+00, -9.29082068e+01,
        -2.73089340e+01, -1.57456917e+00],
       [-3.31272972e-02,  4.96277367e+00, -5.78018953e-01,
         4.45907928e-01, -1.12127831e-01],
       [ 2.66334162e-05, -3.98993037e-03,  4.64710945e-04,
         8.10815068e-03, -1.46753281e-01]])

### STEP 6: Projection onto Basis  
Projection measures how much a vector aligns with another.  
We project player vectors onto orthogonal basis to analyze contributions.

In [30]:
def project_onto_basis(v, ortho_vecs):
    return np.array([np.dot(v, u) / np.dot(u, u) for u in ortho_vecs])  # Calculate the scalar projection coefficients

for i in range(3):
    scalars = project_onto_basis(A[i], ortho_basis)
    print(f"Player {i+1}:", scalars)

Player 1: [ 1.00000000e+00  7.50747229e-14 -1.53904019e-14 -3.28733936e-11
 -2.18344399e-08]
Player 2: [ 1.08187599e+00  1.00000000e+00 -2.27746796e-13 -6.90895571e-11
 -9.87018161e-08]
Player 3: [1.97022071e+00 2.46026433e-01 1.00000000e+00 3.20639488e-11
 8.43004374e-08]


### STEP 7: Least Squares Prediction  
Least squares finds the best approximate solution for inconsistent systems.  
We compute optimal weights to predict player performance scores.

In [31]:
b = (df["Runs"] * 0.4 +
     df["Batting_Avg"] * 0.2 +
     df["Strike_Rate"] * 0.1 +
     df["Wickets"] * 0.2 -
     df["Economy"] * 0.1).values  # Define a target vector 'b' as a weighted sum of player statistics

x_hat = np.linalg.inv(A.T @ A) @ A.T @ b  # Solve the Normal Equation to find the least squares solution
predicted_scores = A @ x_hat  # Map the original features back to the score using the derived weights

df["Performance_Score"] = predicted_scores

df[["Name", "Performance_Score"]].head()

,Name,Performance_Score
0,Player_1,793.81
1,Player_2,875.43
2,Player_3,1554.28
3,Player_4,1086.52
4,Player_5,989.96


### STEP 8: Eigenvalues & Eigenvectors  
Eigenvalues reveal dominant patterns in data.  
We compute covariance matrix and extract eigenvalues to find key trends.

In [32]:
A_centered = A - A.mean(axis=0)  # Subtract the mean of each feature to center the data around the origin
C = (A_centered.T @ A_centered) / (A.shape[0] - 1)  # Compute the covariance matrix of the features

eigenvalues, eigenvectors = np.linalg.eig(C)  # Perform eigen-decomposition to find principal components

print("Eigenvalues:")
print(eigenvalues)

Eigenvalues:
[1.15675399e+06 3.65613016e+03 2.49156956e+02 1.00425340e+02
 3.11834037e+00]


### STEP 9: Dimensionality Reduction  
Reducing dimensions keeps important information while removing noise.  
We project data onto top eigenvectors to simplify the feature space.

In [33]:
k = 2
top_eigvecs = eigenvectors[:, :k]  # Select the first k eigenvectors (principal components)
A_reduced = A_centered @ top_eigvecs  # Project the high-dimensional data onto the lower-dimensional subspace

pd.DataFrame(A_reduced[:10], columns=["Pattern1", "Pattern2"])

,Pattern1,Pattern2
0,318.861631,52.476997
1,157.117280,-58.166084
2,-1569.038558,65.074358
3,-357.977499,-73.033136
4,-171.117039,55.865136
5,-1270.190499,-102.434982
6,26.868972,54.555354
7,-788.142997,59.312352
8,-633.694356,-29.824509
9,265.827767,52.000604


### FINAL OUTPUT: Player Ranking & Analysis  
Applying all linear algebra concepts to real-world insights.  
We rank players, compute performance scores, and analyze patterns.

In [24]:
df_sorted = df.sort_values("Performance_Score", ascending=False)

df_sorted[["Name", "Role", "Performance_Score"]].head(10)

,Name,Role,Performance_Score
58,Player_59,Bowler,1611.98
23,Player_24,All-rounder,1592.93
48,Player_49,Wicketkeeper,1581.59
95,Player_96,All-rounder,1577.94
33,Player_34,Wicketkeeper,1565.35
78,Player_79,Wicketkeeper,1559.43
2,Player_3,Wicketkeeper,1554.28
30,Player_31,All-rounder,1524.87
76,Player_77,All-rounder,1521.45
34,Player_35,Bowler,1514.68
